In [1]:
import pandas as pd

weights = pd.read_csv(
    "data/call_weights_by_zip.csv",
    dtype={"zipcode": "string"},
)

print(f"{len(weights)} ZIPs")
display(weights.head())

42 ZIPs


,zipcode,calls_train,calls_test
0,11201,9532,1898
1,11203,6008,1197
2,11204,2411,466
3,11205,2887,574
4,11206,7272,1392


In [2]:
modzcta = pd.read_csv("data/modzcta.csv", dtype="string")
modzcta.columns = modzcta.columns.str.strip().str.lower()

In [4]:
display(modzcta[["modzcta", "label"]].head(10))

,modzcta,label
0,10001,"10001, 10118"
1,10002,10002
2,10003,10003
3,10026,10026
4,10004,10004
5,10005,10005
6,10006,10006
7,10007,10007
8,10009,10009
9,10010,10010


In [5]:
lookup = modzcta[["modzcta", "label"]].copy()
lookup["zipcode"] = lookup["label"].str.split(",")
lookup = lookup.explode("zipcode")
lookup["zipcode"] = lookup["zipcode"].str.strip()

lookup = (
    lookup.loc[lookup["zipcode"].str.fullmatch(r"\d{5}", na=False),
               ["zipcode", "modzcta"]]
    .dropna(subset=["modzcta"])
    .drop_duplicates()
)

joined = weights.merge(lookup, on="zipcode", how="left", validate="one_to_one")
unmatched = joined.loc[joined["modzcta"].isna()]

print(f"Matched {len(joined) - len(unmatched)} of {len(joined)} ZIPs")
display(unmatched[["zipcode", "calls_train", "calls_test"]])
display(unmatched[["calls_train", "calls_test"]].sum().rename("unmatched_calls"))

Matched 39 of 42 ZIPs


,zipcode,calls_train,calls_test
37,11241,5,1
38,11242,23,1
40,11251,9,3


calls_train    37
calls_test      5
Name: unmatched_calls, dtype: int64

In [6]:
count_cols = ["calls_train", "calls_test"]

area_weights = (
    joined.dropna(subset=["modzcta"])
    .groupby("modzcta", as_index=False)[count_cols]
    .sum()
)

excluded = weights[count_cols].sum() - area_weights[count_cols].sum()
assert excluded.to_dict() == {"calls_train": 37, "calls_test": 5}

print(f"{len(area_weights)} demand areas")
display(area_weights.head())

38 demand areas


,modzcta,calls_train,calls_test
0,11201,9532,1898
1,11203,6008,1197
2,11204,2411,466
3,11205,2887,574
4,11206,7272,1392


In [7]:
matched_zips = joined.dropna(subset=["modzcta"])
zips_per_area = matched_zips.groupby("modzcta")["zipcode"].transform("nunique")

shared = matched_zips.loc[
    zips_per_area.gt(1),
    ["modzcta", "zipcode", "calls_train", "calls_test"],
].sort_values(["modzcta", "zipcode"])

display(shared)

display(
    area_weights.loc[area_weights["modzcta"].isin(shared["modzcta"])]
)

,modzcta,zipcode,calls_train,calls_test
9,11211,11211,3101,606
39,11211,11249,1093,211


,modzcta,calls_train,calls_test
9,11211,4194,817


In [8]:
import geopandas as gpd

boundaries = gpd.read_file("data/modzcta.geojson")
boundaries.columns = boundaries.columns.str.strip().str.lower()
boundaries["modzcta"] = boundaries["modzcta"].astype("string").str.strip()

demand_areas = boundaries[["modzcta", "geometry"]].merge(
    area_weights, on="modzcta", how="inner", validate="one_to_one"
)

assert len(demand_areas) == len(area_weights) == 38
assert demand_areas.geometry.notna().all()
assert not demand_areas.geometry.is_empty.any()

# Pick a point inside each polygon and convert it to longitude and latitude.
points = demand_areas.to_crs("EPSG:2263").representative_point().to_crs("EPSG:4326")

demand = (
    demand_areas.assign(lat=points.y, lon=points.x)
    .rename(columns={"modzcta": "area_id"})
    [["area_id", "lat", "lon", "calls_train", "calls_test"]]
)

display(demand.head())
display(demand.loc[demand["area_id"].eq("11211")])

,area_id,lat,lon,calls_train,calls_test
0,11201,40.694644,-73.990312,9532,1898
1,11203,40.648697,-73.933898,6008,1197
2,11204,40.621012,-73.985932,2411,466
3,11205,40.696517,-73.966369,2887,574
4,11206,40.702414,-73.941854,7272,1392


,area_id,lat,lon,calls_train,calls_test
9,11211,40.71147,-73.957366,4194,817


In [9]:
demand.shape

(38, 5)

In [10]:
demand.to_csv("data/demand.csv", index=False)

In [11]:
# Calculate centroids in a local projected CRS for NYC.
areas_projected = demand_areas.to_crs("EPSG:2263")
centroids = areas_projected.geometry.centroid

# Flag centroids outside their own polygons, then convert to lat/lon.
inside_area = areas_projected.geometry.covers(centroids)
centroids_wgs84 = centroids.to_crs("EPSG:4326")

demand_centroids = (
    demand_areas.assign(lat=centroids_wgs84.y, lon=centroids_wgs84.x)
    .rename(columns={"modzcta": "area_id"})
    [["area_id", "lat", "lon", "calls_train", "calls_test"]]
)

display(demand_centroids.head())
print(f"Centroids outside their area: {(~inside_area).sum()}")
display(demand_centroids.loc[~inside_area, ["area_id", "lat", "lon"]])

,area_id,lat,lon,calls_train,calls_test
0,11201,40.694608,-73.989581,9532,1898
1,11203,40.649574,-73.934362,6008,1197
2,11204,40.618777,-73.984824,2411,466
3,11205,40.694628,-73.966205,2887,574
4,11206,40.701960,-73.942352,7272,1392


Centroids outside their area: 0


,area_id,lat,lon


In [12]:
demand_centroids.shape

(38, 5)

In [13]:
demand_centroids.to_csv("data/demand_centroids.csv", index=False)